In [1]:
import os
import glob
import numpy as np
import pandas as pd

import cv2

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import torchvision.models as models
from torchvision.transforms import transforms

from tqdm import tqdm

DATA_DIR = "../input/plant-pathology-2021-fgvc8"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
SAMPLE_SUB_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test_images")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

test_fram = pd.read_csv(SAMPLE_SUB_CSV)
print(test_fram.head())
print("Num test rows:", len(test_fram))
print("Example image id:", test_fram.values[1][0])




                  image   labels
0  ca6a50c5d2adb8ae.jpg  healthy
1  b686d217a1e2e3a5.jpg  healthy
2  c9a5345ec78b4ac5.jpg  healthy
3  acdc8d88549873d7.jpg  healthy
4  cf7d6b1e5856a021.jpg  healthy
Num test rows: 3727
Example image id: b686d217a1e2e3a5.jpg


In [2]:
class LeafDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform):
        df = pd.read_csv(csv_file)
        self.image_names = df.iloc[:, 0].to_numpy()
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_name = os.path.join(self.root_dir, self.image_names[idx])

        image = cv2.imread(img_name, cv2.IMREAD_COLOR)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {img_name}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        labels = [1]
        sample = {"image": image, "labels": labels}

        if self.transform:
            sample = self.transform(sample)

        return sample


class ToTensor(object):
    def __call__(self, sample):
        image = sample["image"]
        labels = sample["labels"]
        image = np.ascontiguousarray(image.transpose((2, 0, 1)), dtype=np.float32)
        return {"image": torch.from_numpy(image), "labels": labels}


class Rescale(object):
    def __init__(self, output_size):
        assert isinstance(output_size, (int, tuple))
        self.output_size = output_size

    def __call__(self, sample):
        image, labels = sample["image"], sample["labels"]
        h, w = image.shape[:2]

        if isinstance(self.output_size, int):
            if h > w:
                new_h, new_w = self.output_size * h / w, self.output_size
            else:
                new_h, new_w = self.output_size, self.output_size * w / h
        else:
            new_h, new_w = self.output_size

        new_h, new_w = int(new_h), int(new_w)

        img = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        return {"image": img, "labels": labels}


class RandomCrop(object):
    def __init__(self, output_size):
        assert isinstance(output_size, (int, tuple))
        self.output_size = (
            (output_size, output_size) if isinstance(output_size, int) else output_size
        )
        assert len(self.output_size) == 2

    def __call__(self, sample):
        image, labels = sample["image"], sample["labels"]
        h, w = image.shape[:2]
        new_h, new_w = self.output_size

        if h < new_h or w < new_w:
            image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
            return {"image": image, "labels": labels}

        top = (h - new_h) // 2
        left = (w - new_w) // 2
        image = image[top : top + new_h, left : left + new_w]
        return {"image": image, "labels": labels}


class Normalize(object):
    """
    Type-safe normalization: supports both NumPy (HWC) and Torch tensor (CHW).
    Keeps ImageNet normalization (needed for ResNet calibration).
    """

    def __init__(self, mean, std):
        self.mean_np = np.array(mean, dtype=np.float32).reshape(3, 1, 1)
        self.std_np = np.array(std, dtype=np.float32).reshape(3, 1, 1)
        self.mean_t = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std_t = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)

    def __call__(self, sample):
        x = sample["image"]

        if isinstance(x, torch.Tensor):
            x = x.to(dtype=torch.float32)
            x = x * (1.0 / 255.0)
            mean = self.mean_t.to(device=x.device)
            std = self.std_t.to(device=x.device)
            x = (x - mean) / std
            sample["image"] = x
            return sample

        x = x.astype(np.float32) * (1.0 / 255.0)
        x = (x - self.mean_np) / self.std_np
        sample["image"] = x
        return sample


infer_transform = transforms.Compose(
    [
        Rescale(256),
        RandomCrop(224),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

leafDatasets = LeafDataset(
    SAMPLE_SUB_CSV,
    TEST_IMG_DIR,
    transform=infer_transform,
)

print("Dataset item keys:", leafDatasets[0].keys())
print("Image tensor shape:", leafDatasets[0]["image"].shape)
print("Image dtype:", leafDatasets[0]["image"].dtype)



Dataset item keys: dict_keys(['image', 'labels'])
Image tensor shape: torch.Size([3, 224, 224])
Image dtype: torch.float32


In [3]:
batch_size = 64  # Keep as-is.

num_workers = min(8, os.cpu_count() or 2)
pin = torch.cuda.is_available()
test_loader = DataLoader(
    leafDatasets,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    pin_memory=pin,
    persistent_workers=(num_workers > 0),
    prefetch_factor=4 if num_workers > 0 else None,
)



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [4]:
try:
    resnet = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V1)
except Exception:
    resnet = models.resnet152(weights=None)

num_ftrs = resnet.fc.in_features
resnet.fc = nn.Linear(num_ftrs, 6)

print(resnet.fc)



Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth
100%|██████████| 230M/230M [00:02<00:00, 114MB/s]


Linear(in_features=2048, out_features=6, bias=True)


In [5]:
load_path = "../input/modelres/resnet.pkl"
if os.path.exists(load_path):
    state = torch.load(load_path, map_location="cpu")
    resnet.load_state_dict(state)
    print("Loaded checkpoint:", load_path)
else:
    print("Checkpoint not found; using torchvision initialization:", load_path)



Checkpoint not found; using torchvision initialization: ../input/modelres/resnet.pkl


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet.to(device)
resnet.eval()

n_test = len(leafDatasets)
y_pred_logits = np.empty((n_test, 6), dtype=np.float32)

with torch.no_grad():
    stream = tqdm(
        test_loader,
        desc=f"Infer test ({device})",
        total=(n_test + batch_size - 1) // batch_size,
    )
    offset = 0
    for sample in stream:
        X = sample["image"].to(device, non_blocking=True)
        X = X.float()
        pred = resnet(X)  # logits
        bs = pred.shape[0]
        y_pred_logits[offset : offset + bs] = pred.detach().cpu().numpy()
        offset += bs

print("Pred logits shape:", y_pred_logits.shape)



Infer test (cuda): 100%|██████████| 59/59 [00:31<00:00,  1.88it/s]

Pred logits shape: (3727, 6)


In [7]:
LABELS = ["complex", "frog_eye_leaf_spot", "healthy", "powdery_mildew", "rust", "scab"]
label_to_idx = {l: i for i, l in enumerate(LABELS)}


def encode_multilabel_space_delimited(s: str, n_classes: int = 6) -> np.ndarray:
    y = np.zeros((n_classes,), dtype=np.int64)
    if isinstance(s, str) and len(s.strip()) > 0:
        for token in s.split():
            if token in label_to_idx:
                y[label_to_idx[token]] = 1
    return y


def mean_f1_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    eps = 1e-12
    f1s = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c].astype(np.int64)
        yp = y_pred[:, c].astype(np.int64)
        tp = int(((yt == 1) & (yp == 1)).sum())
        fp = int(((yt == 0) & (yp == 1)).sum())
        fn = int(((yt == 1) & (yp == 0)).sum())
        f1 = (2.0 * tp) / (2.0 * tp + fp + fn + eps)
        f1s.append(f1)
    return float(np.mean(f1s))


def apply_thresholds_with_argmax_fallback(
    probs: np.ndarray, thr_vec: np.ndarray
) -> np.ndarray:
    pred_bin = (probs >= thr_vec.reshape(1, -1)).astype(np.int64)
    empty = pred_bin.sum(axis=1) == 0
    if empty.any():
        am = np.argmax(probs, axis=1)
        pred_bin[empty, :] = 0
        pred_bin[empty, am[empty]] = 1
    return pred_bin


train_df = pd.read_csv(TRAIN_CSV)
y_train_true = np.stack(
    [encode_multilabel_space_delimited(x, 6) for x in train_df["labels"].tolist()],
    axis=0,
)

train_dataset_for_thr = LeafDataset(
    TRAIN_CSV,
    TRAIN_IMG_DIR,
    transform=infer_transform,
)

train_loader_for_thr = DataLoader(
    train_dataset_for_thr,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    pin_memory=pin,
    persistent_workers=(num_workers > 0),
    prefetch_factor=4 if num_workers > 0 else None,
)

n_train = len(train_dataset_for_thr)
y_train_logits = np.empty((n_train, 6), dtype=np.float32)

with torch.no_grad():
    stream = tqdm(
        train_loader_for_thr,
        desc=f"Infer train (thr-tune) ({device})",
        total=(n_train + batch_size - 1) // batch_size,
    )
    offset = 0
    for sample in stream:
        X = sample["image"].to(device, non_blocking=True).float()
        pred = resnet(X)
        bs = pred.shape[0]
        y_train_logits[offset : offset + bs] = pred.detach().cpu().numpy()
        offset += bs

train_probs = 1.0 / (1.0 + np.exp(-y_train_logits))


def make_multilabel_stratified_folds(
    y: np.ndarray, n_splits: int = 5, seed: int = 42
) -> np.ndarray:
    """
    Greedy iterative stratification (approximation) implemented locally with NumPy:
    assigns samples to folds to keep per-class counts balanced.
    Returns fold_id for each sample (0..n_splits-1).
    """
    rng = np.random.RandomState(seed)
    n, c = y.shape
    fold_id = -np.ones(n, dtype=np.int64)

    class_counts = y.sum(axis=0).astype(np.float64)
    desired = class_counts / n_splits

    fold_class = np.zeros((n_splits, c), dtype=np.float64)
    fold_size = np.zeros(n_splits, dtype=np.int64)

    card = y.sum(axis=1).astype(np.int64)
    order = np.lexsort((rng.rand(n), -card))

    for i in order:
        yi = y[i].astype(np.float64)

        best_f = None
        best_score = None
        for f in range(n_splits):
            new_fc = fold_class[f] + yi
            score = np.abs(new_fc - desired).sum() + 0.001 * (fold_size[f] + 1)
            if best_score is None or score < best_score:
                best_score = score
                best_f = f

        fold_id[i] = best_f
        fold_class[best_f] += yi
        fold_size[best_f] += 1

    return fold_id


n_splits = 5
fold_id = make_multilabel_stratified_folds(y_train_true, n_splits=n_splits, seed=SEED)
print("Fold sizes:", np.bincount(fold_id, minlength=n_splits).tolist())

thr_grid = np.round(np.arange(0.05, 0.951, 0.01), 2).astype(np.float32)


def coord_descent_fit_thresholds(
    probs_fit: np.ndarray,
    y_fit: np.ndarray,
    probs_eval: np.ndarray,
    y_eval: np.ndarray,
    thr_grid: np.ndarray,
    n_passes: int = 2,
) -> tuple[np.ndarray, float]:
    thr_vec = np.full((y_fit.shape[1],), 0.50, dtype=np.float32)

    def score_on_eval(tv: np.ndarray) -> float:
        pred_eval = apply_thresholds_with_argmax_fallback(probs_eval, tv)
        return mean_f1_score(y_eval, pred_eval)

    base = score_on_eval(thr_vec)
    for _ in range(n_passes):
        improved = False
        for c in range(y_fit.shape[1]):
            best_t = float(thr_vec[c])
            best_sc = base
            thr_try = thr_vec.copy()
            for t in thr_grid:
                thr_try[c] = float(t)
                sc = score_on_eval(thr_try)
                if sc > best_sc + 1e-12:
                    best_sc = sc
                    best_t = float(t)
            if best_t != float(thr_vec[c]):
                improved = True
            thr_vec[c] = best_t
            base = best_sc
        if not improved:
            break

    delta_grid = np.round(np.arange(-0.10, 0.101, 0.01), 2).astype(np.float32)
    best_delta = 0.0
    best_sc = score_on_eval(thr_vec)
    for d in delta_grid:
        thr_try = np.clip(thr_vec + float(d), 0.01, 0.99).astype(np.float32)
        sc = score_on_eval(thr_try)
        if sc > best_sc + 1e-12:
            best_sc = sc
            best_delta = float(d)

    thr_final = np.clip(thr_vec + best_delta, 0.01, 0.99).astype(np.float32)
    final_sc = score_on_eval(thr_final)
    return thr_final, final_sc


oof_pred_true = np.zeros_like(y_train_true, dtype=np.int64)
fold_thresholds = []
fold_scores = []

for f in range(n_splits):
    fit_idx = fold_id != f
    eval_idx = fold_id == f

    thr_f, sc_f = coord_descent_fit_thresholds(
        probs_fit=train_probs[fit_idx],
        y_fit=y_train_true[fit_idx],
        probs_eval=train_probs[eval_idx],
        y_eval=y_train_true[eval_idx],
        thr_grid=thr_grid,
        n_passes=2,
    )
    fold_thresholds.append(thr_f)
    fold_scores.append(sc_f)

    oof_pred_true[eval_idx] = apply_thresholds_with_argmax_fallback(
        train_probs[eval_idx], thr_f
    )

    print(f"Fold {f}: eval mean-F1={sc_f:.6f}, thr={np.round(thr_f, 2).tolist()}")

fold_thresholds = np.stack(fold_thresholds, axis=0)  # (n_splits, 6)

best_thr_per_class = np.median(fold_thresholds, axis=0).astype(np.float32)
best_thr_per_class = np.clip(best_thr_per_class, 0.01, 0.99).astype(np.float32)

oof_score_true = mean_f1_score(y_train_true, oof_pred_true)

print("Aggregated per-class thresholds (median over folds):")
for c, lab in enumerate(LABELS):
    print(f"  {lab:>20s}: thr={best_thr_per_class[c]:.2f}")
print(
    f"Fold eval scores: mean={float(np.mean(fold_scores)):.6f}, std={float(np.std(fold_scores)):.6f}"
)
print(f"TRUE OOF mean-F1 (fold-specific thresholds): {oof_score_true:.6f}")


# CHANGE (score-relevant, minimal): refine the final thresholds by directly maximizing
# TRUE OOF mean-F1 under the exact same decision rule, starting from the median thresholds.
# This keeps the same "per-class threshold + argmax fallback" semantics but typically improves
# generalization vs. a plain fold-median aggregation.
def refine_thresholds_on_oof(
    probs: np.ndarray,
    y_true: np.ndarray,
    thr_init: np.ndarray,
    thr_grid: np.ndarray,
    n_passes: int = 2,
) -> tuple[np.ndarray, float]:
    thr = thr_init.astype(np.float32).copy()

    def score(tv: np.ndarray) -> float:
        pred = apply_thresholds_with_argmax_fallback(probs, tv)
        return mean_f1_score(y_true, pred)

    base = score(thr)
    for _ in range(n_passes):
        improved = False
        for c in range(y_true.shape[1]):
            best_t = float(thr[c])
            best_sc = base
            thr_try = thr.copy()
            for t in thr_grid:
                thr_try[c] = float(t)
                sc = score(thr_try)
                if sc > best_sc + 1e-12:
                    best_sc = sc
                    best_t = float(t)
            if best_t != float(thr[c]):
                improved = True
            thr[c] = best_t
            base = best_sc
        if not improved:
            break
    return thr, float(base)


best_thr_refined, oof_refined_score = refine_thresholds_on_oof(
    probs=train_probs,
    y_true=y_train_true,
    thr_init=best_thr_per_class,
    thr_grid=thr_grid,
    n_passes=2,
)

print("Refined thresholds (optimized on TRUE OOF mean-F1):")
for c, lab in enumerate(LABELS):
    print(f"  {lab:>20s}: thr={best_thr_refined[c]:.2f}")
print(f"TRUE OOF mean-F1 (refined thresholds): {oof_refined_score:.6f}")

# Use refined thresholds for test-time (post-processing only).
best_thr_per_class = best_thr_refined



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Infer train (thr-tune) (cuda): 100%|██████████| 233/233 [02:56<00:00,  1.32it/s]


Fold sizes: [5206, 6453, 3246, 0, 0]
Fold 0: eval mean-F1=0.361627, thr=[0.47999998927116394, 0.5099999904632568, 0.5600000023841858, 0.5199999809265137, 0.3400000035762787, 0.3199999928474426]
Fold 1: eval mean-F1=0.316930, thr=[0.49000000953674316, 0.5, 0.5199999809265137, 0.5199999809265137, 0.33000001311302185, 0.4099999964237213]
Fold 2: eval mean-F1=0.319384, thr=[0.49000000953674316, 0.5, 0.5699999928474426, 0.5199999809265137, 0.05000000074505806, 0.4099999964237213]
Fold 3: eval mean-F1=0.000000, thr=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
Fold 4: eval mean-F1=0.000000, thr=[0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
Aggregated per-class thresholds (median over folds):
               complex: thr=0.49
    frog_eye_leaf_spot: thr=0.50
               healthy: thr=0.52
        powdery_mildew: thr=0.52
                  rust: thr=0.34
                  scab: thr=0.41
Fold eval scores: mean=0.199588, std=0.163736
TRUE OOF mean-F1 (fold-specific thresholds): 0.335746
Refined thresholds (optimized on TRUE

In [8]:
probs = 1.0 / (1.0 + np.exp(-y_pred_logits))

mask = probs >= best_thr_per_class.reshape(1, -1)

indices = []
argmax_idx = np.argmax(probs, axis=1)
for i in range(probs.shape[0]):
    idxs = np.flatnonzero(mask[i]).tolist()
    if len(idxs) == 0:
        idxs = [int(argmax_idx[i])]
    indices.append(idxs)

print("Example predicted indices:", indices[0])



Example predicted indices: [0, 1, 2, 4, 5]


In [9]:
labels = ["complex", "frog_eye_leaf_spot", "healthy", "powdery_mildew", "rust", "scab"]
testlabels = [" ".join(labels[i] for i in idxs) for idxs in indices]

print(testlabels[:5], " ... total:", len(testlabels))



['complex frog_eye_leaf_spot healthy rust scab', 'frog_eye_leaf_spot healthy rust scab', 'complex frog_eye_leaf_spot healthy rust scab', 'complex frog_eye_leaf_spot healthy rust scab', 'frog_eye_leaf_spot healthy rust scab']  ... total: 3727


In [10]:
sub = pd.read_csv(SAMPLE_SUB_CSV)
assert len(sub) == len(
    testlabels
), f"Predictions ({len(testlabels)}) != submission rows ({len(sub)})"
sub["labels"] = testlabels
sub.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", sub.shape)
print(sub.head())

Wrote submission.csv with shape: (3727, 2)
                  image                                        labels
0  ca6a50c5d2adb8ae.jpg  complex frog_eye_leaf_spot healthy rust scab
1  b686d217a1e2e3a5.jpg          frog_eye_leaf_spot healthy rust scab
2  c9a5345ec78b4ac5.jpg  complex frog_eye_leaf_spot healthy rust scab
3  acdc8d88549873d7.jpg  complex frog_eye_leaf_spot healthy rust scab
4  cf7d6b1e5856a021.jpg          frog_eye_leaf_spot healthy rust scab
